In [12]:
import numpy as np
import pandas as pd
import joblib, pickle

from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, mean_absolute_error, 
                            mean_squared_error, r2_score, accuracy_score)
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import LabelEncoder

In [13]:
SEED = 42
np.random.seed(SEED)
pd.set_option('display.max_columns', None)

In [14]:
df_progress = pd.read_csv('../../datas/dataset_userprogress.csv')

In [15]:
df_progress.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1300 entries, 0 to 1299
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   User_ID                   1300 non-null   int64  
 1   Age                       1300 non-null   int64  
 2   Gender                    1300 non-null   object 
 3   Height_cm                 1300 non-null   int64  
 4   Initial_Weight_kg         1300 non-null   int64  
 5   Initial_BMI               1300 non-null   float64
 6   BMI_Category_x            1300 non-null   object 
 7   Body_Fat_Category         1300 non-null   float64
 8   Body_Fat_Percentage_x     1300 non-null   float64
 9   Goal                      1300 non-null   object 
 10  Workout_Frequency         1300 non-null   int64  
 11  Average_Duration_Minutes  1300 non-null   int64  
 12  level                     1300 non-null   object 
 13  Badminton                 1300 non-null   int64  
 14  Football

In [16]:
le_dict = {}
cols_to_encode = {
    'Gender': 'Gender_Encoded',
    'Goal': 'Goal_Encoded',
    'level': 'level_Encoded',
    'BMI_Category_x': 'BMI_Category_x_Encoded',     # Input
    'BMI_Category_y': 'BMI_Category_y_Encoded'      # Output Target
}

for col_name, col_encoded in cols_to_encode.items():
    le = LabelEncoder()
    df_progress[col_encoded] = le.fit_transform(df_progress[col_name])
    le_dict[col_name] = le 

In [17]:
unique_users = df_progress['User_ID'].unique()

# Bagi usernya: 80% user buat latihan, 20% user buat ujian
train_users, test_users = train_test_split(unique_users, test_size=0.2, random_state=42)

In [18]:
# Ambil data baris berdasarkan user yang terpilih
df_train = df_progress[df_progress['User_ID'].isin(train_users)]
df_test = df_progress[df_progress['User_ID'].isin(test_users)]

print(f"Total User: {len(unique_users)}")
print(f"User Latihan: {len(train_users)} | User Ujian: {len(test_users)}")

Total User: 100
User Latihan: 80 | User Ujian: 20


In [19]:
# Definisikan Fitur & Target
# CATATAN: 'User_ID' DIBUANG dari features karena itu cuma nomor absen, tidak mempengaruhi hasil fisik.
features = [
    'Age', 'Gender_Encoded', 'Height_cm', 
    'Initial_Weight_kg', 'Initial_BMI', 'BMI_Category_x_Encoded',
    'Body_Fat_Category', 'Body_Fat_Percentage_x', 
    'Goal_Encoded', 'Workout_Frequency', 'Average_Duration_Minutes', 'level_Encoded',
    'Badminton', 'Football', 'Basketball', 'Tennis', 'Volleyball', 'Table_Tennis', 'Swim',
    'Week'
]

X_train = df_train[features]
X_test = df_test[features]

targets_num = [
    'Weight_kg', 'BMI', 'Body_Fat_Percentage_y', 'Daily_Calories', 
    'Daily_Water_ml', 'Target_Protein_g', 'Target_Carbs_g', 'Target_Fat_g',
    'Limit_Sugar_g', 'Target_Fiber_g', 'Limit_Cholesterol_mg', 
    'Target_Calcium_mg', 'Meal_Frequency'
]
targets_cat = ['BMI_Category_y_Encoded']

In [20]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_absolute_error, accuracy_score, r2_score
import numpy as np

# === 1. AGGRESSIVE PARAMETER GRIDS ===
# We removed '1' from leaf and 'None' from depth to FORCE generalization
reg_params = {
    'n_estimators': [200, 300],
    'max_depth': [5, 10, 15],             # Limited depth prevents complex noise
    'min_samples_split': [10, 20, 30],    # High number prevents specific splits
    'min_samples_leaf': [4, 8, 12],       # Forces grouping of data points
    'max_features': ['sqrt', 'log2']      # Reduces features seen by each tree
}

clf_params = reg_params.copy()

models = {}
SEED = 42

print("\n=== LAPORAN EVALUASI (AGGRESSIVE TUNING) ===")
print(f"{'TARGET':<25} | {'TRAIN':<8} | {'TEST':<8} | {'GAP':<8} | {'BEST PARAMS'}")
print("-" * 90)

# --- 2. Loop Target Angka (Regresi) ---
for t in targets_num:
    rf = RandomForestRegressor(random_state=SEED)
    
    # Randomized Search (Try 15 combinations, 5-fold validation)
    search = RandomizedSearchCV(
        rf, reg_params, n_iter=15, cv=5, scoring='r2', n_jobs=-1, random_state=SEED
    )
    
    search.fit(X_train, df_train[t])
    best_model = search.best_estimator_
    
    # Evaluate
    train_r2 = best_model.score(X_train, df_train[t])
    test_r2 = best_model.score(X_test, df_test[t])
    
    # Save
    models[t] = {'model': best_model, 'type': 'numeric'}
    
    # Display
    gap = train_r2 - test_r2
    params = f"L:{search.best_params_['min_samples_leaf']}/D:{search.best_params_['max_depth']}"
    print(f"{t:<25} | {train_r2*100:.1f}%   | {test_r2*100:.1f}%   | {gap*100:.1f}%   | {params}")

print("-" * 90)

# --- 3. Loop Target Kategori (Klasifikasi) ---
for t in targets_cat:
    rf = RandomForestClassifier(random_state=SEED)
    
    search = RandomizedSearchCV(
        rf, clf_params, n_iter=15, cv=5, scoring='accuracy', n_jobs=-1, random_state=SEED
    )
    
    search.fit(X_train, df_train[t])
    best_model = search.best_estimator_
    
    # Evaluate
    train_acc = best_model.score(X_train, df_train[t])
    test_acc = best_model.score(X_test, df_test[t])
    
    # Save
    models[t] = {'model': best_model, 'type': 'categorical'}
    
    # Display
    gap = train_acc - test_acc
    orig_name = t.replace('_Encoded', '')
    params = f"L:{search.best_params_['min_samples_leaf']}/D:{search.best_params_['max_depth']}"
    print(f"{orig_name:<25} | {train_acc*100:.1f}%   | {test_acc*100:.1f}%   | {gap*100:.1f}%   | {params}")


=== LAPORAN EVALUASI (AGGRESSIVE TUNING) ===
TARGET                    | TRAIN    | TEST     | GAP      | BEST PARAMS
------------------------------------------------------------------------------------------
Weight_kg                 | 99.1%   | 93.6%   | 5.5%   | L:8/D:10
BMI                       | 99.3%   | 94.8%   | 4.5%   | L:8/D:10
Body_Fat_Percentage_y     | 99.1%   | 97.9%   | 1.3%   | L:8/D:10
Daily_Calories            | 99.0%   | 33.9%   | 65.1%   | L:8/D:10
Daily_Water_ml            | 99.2%   | 96.2%   | 3.0%   | L:8/D:15
Target_Protein_g          | 99.2%   | 72.8%   | 26.4%   | L:8/D:15
Target_Carbs_g            | 99.5%   | 84.3%   | 15.2%   | L:4/D:15
Target_Fat_g              | 98.8%   | 76.9%   | 21.9%   | L:8/D:10
Limit_Sugar_g             | 99.0%   | 34.0%   | 65.0%   | L:8/D:10
Target_Fiber_g            | 98.9%   | 35.1%   | 63.9%   | L:8/D:10
Limit_Cholesterol_mg      | 100.0%   | 100.0%   | 0.0%   | L:4/D:15
Target_Calcium_mg         | 100.0%   | 100.0%   | 0.0%  

In [21]:
from sklearn.metrics import mean_absolute_error, accuracy_score, r2_score

# ... (Pastikan data X_train, X_test sudah siap di atas) ...

models = {}

print("\n=== LAPORAN EVALUASI (TRAIN vs TEST) ===")
print(f"{'TARGET':<25} | {'TRAIN Score':<12} | {'TEST Score':<12} | {'GAP':<8} | {'INFO'}")
print("-" * 80)

# --- 1. Loop Target Angka (Regresi) ---
rf_angka = RandomForestRegressor(n_estimators=100, random_state=SEED)
for t in targets_num:
    rf_angka.fit(X_train, df_train[t])
    
    # Hitung Skor Latihan (Train)
    y_train_pred = rf_angka.predict(X_train)
    train_r2 = r2_score(df_train[t], y_train_pred) # Score 0.0 - 1.0
    
    # Hitung Skor Ujian (Test)
    y_test_pred = rf_angka.predict(X_test)
    test_r2 = r2_score(df_test[t], y_test_pred)   # Score 0.0 - 1.0
    test_mae = mean_absolute_error(df_test[t], y_test_pred)
    
    # Simpan Model
    models[t] = {'model': rf_angka, 'type': 'numeric'}
    
    # Tampilkan (Gap adalah selisih score)
    gap = train_r2 - test_r2
    print(f"{t:<25} | {train_r2*100:.1f}%       | {test_r2*100:.1f}%       | {gap*100:.1f}%   | Meleset +/- {test_mae:.2f}")

print("-" * 80)

# --- 2. Loop Target Kategori (Klasifikasi) ---
rf_kategori = RandomForestClassifier(n_estimators=100, random_state=SEED)
for t in targets_cat:
    rf_kategori.fit(X_train, df_train[t])
    
    # Hitung Skor Latihan
    y_train_pred = rf_kategori.predict(X_train)
    train_acc = accuracy_score(df_train[t], y_train_pred)
    
    # Hitung Skor Ujian
    y_test_pred = rf_kategori.predict(X_test)
    test_acc = accuracy_score(df_test[t], y_test_pred)
    
    # Simpan Model
    models[t] = {'model': rf_kategori, 'type': 'categorical'}
    
    # Tampilkan
    gap = train_acc - test_acc
    orig_name = t.replace('_Encoded', '')
    print(f"{orig_name:<25} | {train_acc*100:.1f}%       | {test_acc*100:.1f}%       | {gap*100:.1f}%   | Akurasi")

# ... (Lanjut simpan ke pickle di bawahnya) ...


=== LAPORAN EVALUASI (TRAIN vs TEST) ===
TARGET                    | TRAIN Score  | TEST Score   | GAP      | INFO
--------------------------------------------------------------------------------
Weight_kg                 | 100.0%       | 99.1%       | 0.9%   | Meleset +/- 1.78
BMI                       | 100.0%       | 97.9%       | 2.1%   | Meleset +/- 0.91
Body_Fat_Percentage_y     | 100.0%       | 99.6%       | 0.4%   | Meleset +/- 0.77
Daily_Calories            | 100.0%       | 20.4%       | 79.6%   | Meleset +/- 190.67
Daily_Water_ml            | 100.0%       | 95.6%       | 4.4%   | Meleset +/- 124.60
Target_Protein_g          | 100.0%       | 60.5%       | 39.5%   | Meleset +/- 16.43
Target_Carbs_g            | 100.0%       | 85.8%       | 14.2%   | Meleset +/- 20.28
Target_Fat_g              | 100.0%       | 87.4%       | 12.6%   | Meleset +/- 3.92
Limit_Sugar_g             | 100.0%       | 24.1%       | 75.9%   | Meleset +/- 4.65
Target_Fiber_g            | 100.0%       | 14

In [22]:
data_progress = {
    'models_dict': models,     # Isinya model + info tipenya
    'encoders': le_dict,       # Isinya semua encoder (Gender, Goal, dll)
    'features': features       # Urutan fitur input (PENTING)
}

with open('../../models/model_progress.pickle', 'wb') as f:
    pickle.dump(data_progress, f)

print("\n 'model_progress.pickle' telah disimpan.")


 'model_progress.pickle' telah disimpan.
